In [ ]:
!pip install transformers datasets librosa torchaudio accelerate matplotlib captum

In [ ]:
import torch
import numpy as np
from transformers import TrainingArguments, Trainer
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

repo_id = "VasilisAsim/wav2vec-finetuned-TESS"

checkpoints_all = [
    {"name": "pretrained", "source": "facebook/wav2vec2-base", "revision": None},
    {"name": "finetuned",  "source": repo_id, "revision": None},
]

def load_checkpoint_for_probing(source, revision=None):
    if revision is not None:
        model = AutoModelForAudioClassification.from_pretrained(source, revision=revision)
        feature_extractor = AutoFeatureExtractor.from_pretrained(source, revision=revision)
    else:
        model = AutoModelForAudioClassification.from_pretrained(source)
        feature_extractor = AutoFeatureExtractor.from_pretrained(source)
    model.to(device).eval()
    return model, feature_extractor

In [ ]:
from datasets import load_dataset, Audio

dataset = load_dataset("AbstractTTS/TESS", split="train")
dataset = dataset.cast_column("audio", Audio(sampling_rate=16_000))

In [ ]:
def add_speaker(example):
    example["speaker"] = example["file"].split("_")[0]  # "OAF" ή "YAF"
    return example

dataset = dataset.map(add_speaker)

train_dataset = dataset.filter(lambda x: x["speaker"] == "OAF")
test_dataset  = dataset.filter(lambda x: x["speaker"] == "YAF")

print(f"Train (OAF): {len(train_dataset)} δείγματα")
print(f"Test (YAF): {len(test_dataset)} δείγματα")

In [ ]:
import torch
import numpy as np
from sklearn.model_selection import train_test_split

def extract_all_layers(model, feature_extractor, dataset, batch_size=8):
    """
    Επιστρέφει:
    - hidden_states_per_layer: list από np.array [num_samples, hidden_dim], ένα ανά layer
    - labels: np.array με τα labels
    """
    all_labels = []
    layer_features = None

    for i in range(0, len(dataset), batch_size):
        batch = dataset[i:i+batch_size]
        audios = [a["array"] for a in batch["audio"]]

        inputs = feature_extractor(
            audios, sampling_rate=16000, return_tensors="pt", padding=True
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)

        hidden_states = outputs.hidden_states

        if layer_features is None:
            layer_features = [[] for _ in range(len(hidden_states))]

        for layer_idx, layer_out in enumerate(hidden_states):
            pooled = layer_out.mean(dim=1)
            layer_features[layer_idx].append(pooled.cpu().numpy())

        all_labels.extend(batch["emotion"])

    layer_features = [np.concatenate(layer, axis=0) for layer in layer_features]
    return layer_features, np.array(all_labels)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import (
    accuracy_score, f1_score, balanced_accuracy_score, roc_auc_score
)
import pandas as pd

def evaluate_layer(X_train, y_train, X_test, y_test, num_classes):
    probe = LogisticRegression(max_iter=2000, multi_class="multinomial")
    probe.fit(X_train, y_train)

    y_pred = probe.predict(X_test)
    y_proba = probe.predict_proba(X_test)

    acc = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average="macro")
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    uar = bal_acc

    # Macro AUC (One-vs-Rest) - χρειάζεται one-hot ground truth
    y_test_bin = label_binarize(y_test, classes=range(num_classes))
    try:
        auc = roc_auc_score(y_test_bin, y_proba, average="macro", multi_class="ovr")
    except ValueError:
        auc = np.nan

    return {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "UAR": uar,
        "balanced_accuracy": bal_acc,
        "macro_auc": auc
    }


def run_layer_wise_probing(model, feature_extractor, train_ds, test_ds):
    le = LabelEncoder()
    all_labels_train = train_ds["emotion"]
    le.fit(all_labels_train)
    num_classes = len(le.classes_)

    train_layers, train_labels_raw = extract_all_layers(model, feature_extractor, train_ds)
    test_layers, test_labels_raw = extract_all_layers(model, feature_extractor, test_ds)

    y_train = le.transform(train_labels_raw)
    y_test = le.transform(test_labels_raw)

    results = []
    for layer_idx in range(len(train_layers)):
        metrics = evaluate_layer(
            train_layers[layer_idx], y_train,
            test_layers[layer_idx], y_test,
            num_classes
        )
        metrics["layer"] = layer_idx
        results.append(metrics)

    return pd.DataFrame(results)

In [ ]:
all_results = []

for ckpt in checkpoints_all:
    print(f"--- Probing: {ckpt['name']} ---")
    model_ckpt, fe_ckpt = load_checkpoint_for_probing(ckpt["source"], revision=ckpt["revision"])

    df = run_layer_wise_probing(model_ckpt, fe_ckpt, train_dataset, test_dataset)
    df["checkpoint"] = ckpt["name"]
    all_results.append(df)

    del model_ckpt
    torch.cuda.empty_cache()

results_all = pd.concat(all_results, ignore_index=True)

# Plotting όλων των μετρικών

In [ ]:
metrics_to_plot = ["accuracy", "macro_f1", "UAR", "macro_auc"]
selected_checkpoints = ["pretrained", "finetuned"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, metric in zip(axes.flat, metrics_to_plot):
    for ckpt_name in selected_checkpoints:
        group = results_all[results_all["checkpoint"] == ckpt_name]
        ax.plot(group["layer"], group[metric], marker="o", label=ckpt_name)
    ax.set_title(metric)
    ax.set_xlabel("Layer")
    ax.grid(alpha=0.3)

handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.02))

plt.tight_layout()
plt.show()